# 5장 · 확률변수와 확률분포 🎲

웹 챕터 **"확률변수와 확률분포"** 와 짝을 이루는 Python 실습입니다.
기댓값을 손으로 계산한 값과 시뮬레이션 평균이 정말 일치하는지, 이항분포가 어떻게 생겼는지 직접 확인합니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch5')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# 한글 폰트 (Windows) — 이게 없으면 그래프의 한글이 깨져요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

rng = np.random.default_rng(42)  # 시드 고정 — 실행할 때마다 같은 결과

## 1. 확률변수: 동전 2개의 앞면 수

동전 2개를 던져 **앞면의 개수**를 X 라고 하면 X 는 0, 1, 2 중 하나입니다.
이론값은 P(X=0)=1/4, P(X=1)=1/2, P(X=2)=1/4입니다. 시뮬레이션으로 확인해 봅시다.

In [ ]:
# 동전 2개 던지기를 100,000번 반복 (0=뒷면, 1=앞면)
flips = rng.integers(0, 2, size=(100_000, 2))
X = flips.sum(axis=1)  # 앞면의 개수 — 이것이 확률변수!

theory = [0.25, 0.5, 0.25]
for k in range(3):
    print(f'P(X={k})  시뮬레이션: {np.mean(X == k):.4f}   이론값: {theory[k]}')

## 2. 기댓값: 확률로 가중평균한 값

$$E[X] = \sum x \cdot p(x)$$

주사위 한 개의 기댓값은 (1+2+…+6)/6 = **3.5**. "오래 반복했을 때의 평균"이라는 뜻을 직접 확인합니다.

In [ ]:
values = np.arange(1, 7)
probs = np.full(6, 1 / 6)
print('공식으로 구한 기댓값 E[X] =', np.sum(values * probs))

# 정말 '오래 반복한 평균'일까? 주사위를 점점 많이 굴려보면...
for n in [10, 100, 10_000, 1_000_000]:
    rolls = rng.integers(1, 7, size=n)
    print(f'{n:>9,}번 굴린 평균: {rolls.mean():.4f}')

굴리는 횟수가 늘수록 평균이 3.5 에 달라붙습니다.

### 복권의 기대 상금 (웹 챕터의 예)

1000원짜리 복권: 1등 10만 원(확률 0.001), 2등 1만 원(확률 0.01), 나머지 꽝.

E = 100000×0.001 + 10000×0.01 = **200원**. 가격(1000원)의 5분의 1이에요.

In [ ]:
prize = np.array([100_000, 10_000, 0])
p = np.array([0.001, 0.01, 0.989])
print('복권 기대 상금:', np.sum(prize * p), '원  (가격 1000원과 비교!)')

# 복권을 100만 장 산다면? 평균 상금은 기댓값에 붙는다
tickets = rng.choice(prize, size=1_000_000, p=p)
print('100만 장 시뮬레이션 평균 상금:', round(tickets.mean(), 1), '원')

## 3. 이항분포 B(n, p)

성공 확률 p 인 시행을 n 번 반복했을 때 **성공 횟수**의 분포입니다.

$$P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$$

공식 계산은 `scipy.stats.binom` 이 대신해 줍니다.

In [ ]:
n, p = 10, 0.5
k = np.arange(n + 1)
pmf = stats.binom.pmf(k, n, p)

plt.figure(figsize=(7, 3.5))
plt.bar(k, pmf, color='#6366f1')
plt.title(f'이항분포 B({n}, {p}): 동전 10번 중 앞면 횟수')
plt.xlabel('앞면 횟수 k')
plt.ylabel('P(X=k)')
plt.show()

# 웹 챕터의 손계산 검산: 동전 3번 중 앞면 2번 = 3/8
print('P(X=2), B(3, 0.5) =', stats.binom.pmf(2, 3, 0.5), ' (= 3/8 = 0.375)')
# 연습문제의 검산: 동전 4번 중 앞면 2번 = 6/16 (우연히 같은 0.375!)
print('P(X=2), B(4, 0.5) =', stats.binom.pmf(2, 4, 0.5), ' (= 6/16 = 0.375)')

## 4. n 을 키우면 종 모양이 나타난다

웹의 '이항분포 탐험기'에서 본 것으로, 6장(정규분포)의 예고편입니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, n in zip(axes, [5, 20, 50]):
    k = np.arange(n + 1)
    ax.bar(k, stats.binom.pmf(k, n, 0.5), color='#6366f1')
    ax.set_title(f'B({n}, 0.5)')
fig.suptitle('n 이 커질수록 좌우대칭 종 모양으로!')
plt.tight_layout()
plt.show()

## 5. E[X] = np, Var(X) = np(1−p) 확인

"한 번에 평균 p 씩, n 번이면 np". 공식·scipy·시뮬레이션 세 가지가 일치하는지 봅니다.

In [ ]:
n, p = 100, 0.3
print('공식        E =', n * p, '   Var =', n * p * (1 - p))
print('scipy       E =', stats.binom.mean(n, p), '   Var =', stats.binom.var(n, p))

samples = rng.binomial(n, p, size=200_000)
print('시뮬레이션  E =', round(samples.mean(), 2), '  Var =', round(samples.var(), 2))

## 정리

- **확률변수** = 시행 결과에 숫자를 붙인 것 · **확률분포** = 값마다 확률 (합계 1)
- **기댓값 E[X] = Σ x·p(x)** = 확률로 가중평균 = 아주 오래 반복했을 때의 평균
- **이항분포 B(n, p)**: n 번 중 성공 횟수. E[X] = np, Var(X) = np(1−p)
- n 이 커지면 종 모양 → 다음 챕터 **정규분포**에서 이 곡선의 정체를 밝힙니다

다음 노트북: `ch06_normal_distribution.ipynb`